In [ ]:
pgf_backend = True
# pgf_backend = False
figure_str = "paper_"

import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append("../../src/")
import pickle

from lightning import seed_everything

from model_evaluation.trading_strategies import (
    trading_fixed_hours,
    trading_optimal_bids,
)
from model_training.data_modules.utils import EPFDataModule
from model_evaluation.utils import (
    compute_crps,
    compute_mpiw,
    compute_picp,
    compute_pinball,
    compute_pinball_naive,
)


In [ ]:
if pgf_backend:
    mpl.use("pgf")

In [ ]:
# IEEE Access matplotlib settings
# Source: https://journals.ieeeauthorcenter.ieee.org/create-your-ieee-journal-article/create-graphics-for-your-article/resolution-and-size/

font_size = 8

plt.rcParams.update(
    {
        "pgf.rcfonts": False,  # don't use matplotlib defaults
        "pgf.texsystem": "lualatex",  # use LuaLaTeX
        "font.family": "serif",
        "font.serif": ["Times New Roman"],  # system Times New Roman
        "figure.dpi": 300,
        "font.size": font_size,
        "axes.titlesize": font_size,
        "figure.titlesize": font_size,
        "axes.labelsize": font_size,
        "legend.fontsize": font_size,
        "xtick.labelsize": font_size,
        "ytick.labelsize": font_size,
        "pgf.preamble": r"\usepackage{fontspec}\setmainfont{Times New Roman}",
    }
)

# IEEE standard widths (inches)
linewidth_singlecol = 3.5  # single-column figure
linewidth_doublecol = 7.16  # double-column figure
max_height = 9.25  # max text height

golden_ratio = (5**0.5 - 1) / 2

dpi = 300


def set_size(width=linewidth_singlecol, ratio=golden_ratio, height_pad=0):
    height = width * ratio + height_pad
    return (width, min(height, max_height))

In [ ]:
seed_everything(0)


In [ ]:
file_names = [
    "NaiveHS",
    "DDNN_Ens",
    "MCD",
    "EvDNN",
    "DDNN_CP",
    "Ens_CP",
    "MCD_CP",
    "EvDNN_CP",
    "LEAR_GARCH",
    "LEAR_QRA",
    "LEAR_CP",
    "XGBoost_GARCH",
    "XGBoost_QRA",
    "XGBoost_CP",
]
models_dict = []

In [ ]:
for file_name in file_names:
    with open(
        f"../evaluate_models/results/metric_evaluation/{file_name}.pkl",
        "rb",
    ) as f:
        models_dict.extend(pickle.load(f))

In [ ]:
# Adjust the model names for plotting
for model in models_dict:
    if model["model_name"] == "ddnn_normal":
        model["model_name"] = "DDNN"
    elif model["model_name"] == "ens5_normal":
        model["model_name"] = "Ens5"
    elif model["model_name"] == "ens10_normal":
        model["model_name"] = "Ens10"
    elif model["model_name"] == "mcd10_normal":
        model["model_name"] = "MCD10"
    elif model["model_name"] == "mcd30_normal":
        model["model_name"] = "MCD30"
    elif model["model_name"] == "naive_hs_train_normal":
        model["model_name"] = "Naive-HS$_{train}$"
    elif model["model_name"] == "naive_hs_val_normal":
        model["model_name"] = "Naive-HS$_{val}$"
    elif model["model_name"] == "evdnn_normal":
        model["model_name"] = "EvDNN"
    elif model["model_name"] == "lasso_garch":
        model["model_name"] = "Lasso-GARCH"
    elif model["model_name"] == "LEAR_GARCH":
        model["model_name"] = "LEAR-GARCH"
    elif model["model_name"] == "LEAR_QRA":
        model["model_name"] = "LEAR-QRA"
    elif model["model_name"] == "XGBoost_GARCH":
        model["model_name"] = "XGBoost-GARCH"
    elif model["model_name"] == "XGBoost_QRA":
        model["model_name"] = "XGBoost-QRA"


In [ ]:
models_to_exclude = {
    "Ens5",
    "MCD10",
    "Naive-HS$_{train}$",
    "EvDNN",
    "EvDNN-CP",
    #"MCD30-CP",
    #"Ens10-CP",
}
models_dict = [
    model for model in models_dict if model["model_name"] not in models_to_exclude
]

In [ ]:
quantiles = np.linspace(0.01, 0.99, 99)
confidence_levels = np.flip(
    np.array([quantiles[-i - 1] - quantiles[i] for i in range(len(quantiles) // 2)])
)
confidence_levels

In [ ]:
n_runs = 10
standardization_case = "mean_std"

In [ ]:
# Load the data
val_date = "2022-12-01"
test_date = "2023-12-01"
end_date = "2024-11-30"
data_file_path = "../../data/processed/smard_data_201810010000_202501010000.npz"
data_module = EPFDataModule(
    data_file_path=data_file_path,
    val_date=val_date,
    test_date=test_date,
    end_date=end_date,
    batch_size=32,
    standardization_case=standardization_case,
)

train_input, train_labels = data_module.train_dataset[:]
val_input, val_labels = data_module.val_dataset[:]
test_input, test_labels = data_module.test_dataset[:]

data_input, data_labels = test_input, test_labels

data_labels = data_labels * data_module.scale_target + data_module.offset_target
data_labels = data_labels.detach().numpy()

In [ ]:
for model in models_dict:
    model["pinball"] = compute_pinball(quantiles, model["quantile"], data_labels)
    # model["pinball_naive"] = compute_pinball_naive(quantiles, model["quantile"], data_labels)
    model["crps_all"] = np.mean(model["pinball"], axis=1)
    # model["crps_all_naive"] = np.mean(model["pinball_naive"], axis=1)

In [ ]:
# for model in models_dict:
#     print(model["model_name"], model["pinball"].mean())
#     print(model["model_name"], model["pinball_naive"].mean())

In [ ]:
from scipy.stats import norm
import numpy as np
import pandas as pd


def dm_test(errors1, errors2, alternative="two-sided"):
    """
    Simple Diebold-Mariano test for comparing forecast accuracy.

    Parameters:
    -----------
    errors1, errors2 : array-like
        Forecast errors (or loss values) from two models
    alternative : str, optional
        Alternative hypothesis. Options:
        - 'two-sided' (default): H1: d != 0
        - 'greater': H1: d > 0 (errors1 > errors2, model2 is better)
        - 'less': H1: d < 0 (errors1 < errors2, model1 is better)

    Returns:
    --------
    dm_stat : float
        DM test statistic
    p_value : float
        p-value based on the specified alternative hypothesis
    """
    # Convert to numpy arrays
    errors1 = np.array(errors1)
    errors2 = np.array(errors2)

    # Calculate loss differential
    d = errors1 - errors2

    # DM statistic
    d_mean = np.mean(d)
    d_var = np.var(d, ddof=1)
    n = len(d)

    dm_stat = d_mean / np.sqrt(d_var / n)

    # Calculate p-value based on alternative hypothesis
    if alternative == "two-sided":
        p_value = 2 * (1 - norm.cdf(np.abs(dm_stat)))
    elif alternative == "greater":
        p_value = 1 - norm.cdf(dm_stat)
    elif alternative == "less":
        p_value = norm.cdf(dm_stat)
    else:
        raise ValueError("alternative must be 'two-sided', 'greater', or 'less'")

    return dm_stat, p_value


def dm_test_n_runs(errors1, errors2, alternative="two-sided", mode="all"):
    dm_stat_list = []
    p_values_list = []
    
    assert errors1.shape[0] == errors2.shape[0], "Both error arrays must have the same number of runs"

    if mode == "all":
        for i in range(errors1.shape[0]):
            for j in range(errors2.shape[0]):
                dm_stat, p_value = dm_test(errors1[i], errors2[j], alternative=alternative)
                dm_stat_list.append(dm_stat)
                p_values_list.append(p_value)
    elif mode == "runs":
        for i in range(errors1.shape[0]):
            dm_stat, p_value = dm_test(errors1[i], errors2[i], alternative=alternative)
            dm_stat_list.append(dm_stat)
            p_values_list.append(p_value)
    else:
        raise ValueError("mode must be 'all' or 'runs'")


    dm_stat_stack = np.stack(dm_stat_list)
    p_values_stack = np.stack(p_values_list)
    return dm_stat_stack, p_values_stack

In [ ]:
# Simple example: Compare first two models
model1 = models_dict[0]["model_name"]
model2 = models_dict[1]["model_name"]

dm_stat, p_value = dm_test(models_dict[0]["crps_all"][0], models_dict[1]["crps_all"][0], alternative="less")

print(f"{model1} vs {model2}")
print(f"DM statistic: {dm_stat:.3f}")
print(f"p-value: {p_value:.4f}")
print(f"Significant: {'Yes' if p_value < 0.05 else 'No'}")

In [ ]:
# Simple example: Compare first two models
model1 = models_dict[0]["model_name"]
model2 = models_dict[1]["model_name"]

dm_stat, p_value = dm_test_n_runs(
    models_dict[0]["crps_all"], models_dict[1]["crps_all"], alternative="less", mode="all"
)

print(f"{model1} vs {model2}")
print(f"DM statistic: {dm_stat.mean():.3f}")
print(f"p-value: {p_value.mean():.4f}")
print(f"Significant: {'Yes' if p_value.mean() < 0.05 else 'No'}")

In [ ]:
dm_stat

In [ ]:
p_value

In [ ]:
def calculate_dm_stat_and_p_value_table(models_dict, alternative="two-sided", mode="all"):
    n_models = len(models_dict)
    dm_stat_matrix_mean = np.zeros((n_models, n_models))
    dm_stat_matrix_std = np.zeros((n_models, n_models))
    p_value_matrix_mean = np.zeros((n_models, n_models))
    p_value_matrix_std = np.zeros((n_models, n_models))

    for i in range(n_models):
        for j in range(n_models):
            if i != j:
                dm_stat, p_value = dm_test_n_runs(
                    models_dict[i]["crps_all"], models_dict[j]["crps_all"], alternative=alternative, mode=mode
                )
                dm_stat_matrix_mean[i, j] = dm_stat.mean()
                dm_stat_matrix_std[i, j] = dm_stat.std()
                p_value_matrix_mean[i, j] = p_value.mean()
                p_value_matrix_std[i, j] = p_value.std()
            else:
                dm_stat_matrix_mean[i, j] = np.nan
                dm_stat_matrix_std[i, j] = np.nan
                p_value_matrix_mean[i, j] = np.nan
                p_value_matrix_std[i, j] = np.nan

    model_names = [model["model_name"] for model in models_dict]
    dm_stat_df_mean = pd.DataFrame(
        dm_stat_matrix_mean, index=model_names, columns=model_names
    )
    dm_stat_df_std = pd.DataFrame(
        dm_stat_matrix_std, index=model_names, columns=model_names
    )
    p_value_df_mean = pd.DataFrame(
        p_value_matrix_mean, index=model_names, columns=model_names
    )
    p_value_df_std = pd.DataFrame(
        p_value_matrix_std, index=model_names, columns=model_names
    )
    return dm_stat_df_mean, dm_stat_df_std, p_value_df_mean, p_value_df_std

In [ ]:
dm_stat_df_mean, dm_stat_df_std, p_value_df_mean, p_value_df_std = (
    calculate_dm_stat_and_p_value_table(models_dict, alternative="less")
)

In [ ]:
dm_stat_df_mean

In [ ]:
dm_stat_df_std

In [ ]:
p_value_df_mean

In [ ]:
p_value_df_std

In [ ]:
print("dm_stat_df_mean")
print(dm_stat_df_mean.to_latex(float_format="%.3f", na_rep="--"))
print("dm_stat_df_std")
print(dm_stat_df_std.to_latex(float_format="%.3f", na_rep="--"))
print("p_value_df_mean")
print(p_value_df_mean.to_latex(float_format="%.3f", na_rep="--"))
print("p_value_df_std")
print(p_value_df_std.to_latex(float_format="%.3f", na_rep="--"))

In [ ]:
# Create a plot for p-values mean from DM test
import matplotlib.colors as colors

# Get model names and create the matrix
model_names = [model["model_name"] for model in models_dict]
num = len(model_names)
dm_matrix = p_value_df_mean.values

# Create colorblind-friendly colormap with discontinuity at 0.05
plt.figure(figsize=set_size(height_pad=1))
n = 50
# blues = plt.cm.Blues_r(np.linspace(0.25, 0.75, n))
# oranges = plt.cm.Oranges(np.linspace(0.25, 0.75, n))
blues = plt.cm.Blues_r(np.linspace(0.2, 0.7, n))
oranges = plt.cm.Oranges(np.linspace(0.3, 0.8, n))
rgb_color_map = colors.ListedColormap(np.vstack([blues, oranges]))
rgb_color_map.set_over('black')  # Values > 0.1 appear black

plt.imshow(dm_matrix.astype(float), cmap=rgb_color_map, vmin=0, vmax=0.1)
plt.plot(range(dm_matrix.shape[0]), range(dm_matrix.shape[0]), "wx", markersize=8)

# Add grid lines between cells
for i in range(num + 1):
    plt.axhline(i - 0.5, color="white", linewidth=0.4, alpha=0.4)
    plt.axvline(i - 0.5, color="white", linewidth=0.4, alpha=0.4)

# Add colorbar with proper labels
cbar = plt.colorbar()
cbar.set_label("p-value", rotation=270, labelpad=15)

plt.xticks(range(num), model_names, rotation=45.0, ha="right")
plt.yticks(range(num), model_names, rotation=0.0)
# plt.title("Diebold-Mariano Test: Mean p-values")
plt.tight_layout()

# Save the plot
folder_path = "./plots/"
plt.savefig(folder_path + "dm_test_pair_pvalues_mean.pdf", dpi=300, bbox_inches="tight")
plt.show()